# DiffGR Replication: MovieLens 1M (ML-1M)

This notebook replicates the experiments from the ECIR 2025 research paper:
> **"DiffGR: A Discrete Diffusion-Based Model for Personalised Recommendation by Reconstructing User-Item Bipartite Graphs"**  
> Zheng Ju, Honghui Du, Elias Tragos, Neil Hurley, Aonghus Lawlor (Insight Centre for Data Analytics, UCD)

### Paper Targets (Table 1):
- **Recall@20:** 0.2397
- **NDCG@20:** 0.3413

In [ ]:
import sys
import os
sys.path.append("..")

import time
import torch
import torch.optim as optim

from src.dataset import DataLoaderHandler
from src.diffusion import DiscreteDiffusion
from src.model import DiffGRModel
from src.metrics import evaluate_model

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# 1. Load ML-1M (70% train / 10% validation / 20% test)
data_dir = "../Data"
data_handler = DataLoaderHandler(data_dir=data_dir, dataset_name="ml-1m", min_interactions=5, seed=42)
train_loader = data_handler.get_train_loader(batch_size=256, shuffle=True)

In [ ]:
# 2. Initialize Model & Discrete Diffusion (T=5, s=0.08 as specified in paper)
model = DiffGRModel(
    n_users=data_handler.n_users,
    n_items=data_handler.n_items,
    emb_dim=64,
    hidden_dim=256,
    dropout=0.1
).to(device)

diffusion = DiscreteDiffusion(steps=5, s=0.08, device=device)
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
print("Model and Discrete Diffusion initialized successfully.")

In [ ]:
# 3. Train DiffGR Model
epochs = 20
eval_interval = 5

print(f"Starting training for {epochs} epochs...")
start_time = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    batches = 0
    for u_ids, g0 in train_loader:
        u_ids = u_ids.to(device)
        g0 = g0.to(device)
        
        optimizer.zero_grad()
        loss = diffusion.compute_elbo_loss(model, u_ids, g0)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        batches += 1
    
    avg_loss = epoch_loss / max(1, batches)
    if epoch % eval_interval == 0 or epoch == epochs:
        r20, n20 = evaluate_model(model, data_handler, split="val", k=20, device=device)
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Loss: {avg_loss:.4f} | Val Recall@20: {r20:.4f} | Val NDCG@20: {n20:.4f}")
    else:
        print(f"Epoch [{epoch:02d}/{epochs:02d}] Loss: {avg_loss:.4f}")

elapsed = (time.time() - start_time) / 60.0
print(f"Training finished in {elapsed:.2f} minutes.")

In [ ]:
# 4. Final Evaluation on Test Set
print("Evaluating on Test Set (K=20)...")
test_r20, test_n20 = evaluate_model(model, data_handler, split="test", k=20, device=device)

print("=" * 50)
print("DIFFGR ML-1M RESULTS (Top-20):")
print(f"Replicated Recall@20: {test_r20:.4f}  |  Paper Reported: 0.2397")
print(f"Replicated NDCG@20:   {test_n20:.4f}  |  Paper Reported: 0.3413")
print("=" * 50)